In [126]:
import jax.numpy as jnp
import jax

# 4 rooms with simple x,y coordinates.
# features x, y, xy, and bias

Phi = jnp.array([
    [1,1,1,1], # room 1 (bottom left)
    [1,2,2,1], # top left
    [2,1,2,1], # bottom right
    [2,2,4,1] # room 4 (top right)
])
# Phi /= jnp.linalg.norm(Phi, axis=1, keepdims=True)
N,k = Phi.shape
visits = jnp.array([8, 1, 2, 0]) * 10
T = jnp.sum(visits)
vist_mat = jnp.diag(visits)
dist_mat = visits / T
dist_mat

indices = jnp.repeat(jnp.arange(len(visits)), visits)

data = Phi[indices]

gram = jnp.sum(jax.vmap(jnp.outer)(data, data), axis=0)
gram = 1/2 * gram +  1/2 * gram.T

ε = 0.1
precision = jnp.linalg.inv(gram + jnp.eye(gram.shape[0]) + ε) 
r = []
for i in range(N): # for each state:
    M = Phi[i].T @ precision @ Phi[i]
    r.append(jnp.sqrt(M))
    print(f'state {i} \n ---------')
    print('effective N is ' , 1/M)
    print('N(s) is, ', visits[i])
    print('M-Distance is ' , M)
    print('bonus is ' , jnp.sqrt(M))
    print('\n ---------')

state 0 
 ---------
effective N is  84.538185
N(s) is,  80
M-Distance is  0.011828974
bonus is  0.10876109

 ---------
state 1 
 ---------
effective N is  10.97344
N(s) is,  10
M-Distance is  0.091129124
bonus is  0.301876

 ---------
state 2 
 ---------
effective N is  20.973408
N(s) is,  20
M-Distance is  0.047679424
bonus is  0.21835619

 ---------
state 3 
 ---------
effective N is  1.649811
N(s) is,  0
M-Distance is  0.60613
bonus is  0.77854353

 ---------


$$
G = \sum_s N(s) \phi(s) \phi(s)^\top$$
$$
x^\top G^{-1} x \leq \frac{1}{\sum_s N(s) x^\top \phi(s)}
$$

In [127]:
for i in range(N):
    N_eff = 1/(r[i]**2) # ri = sqrt(1/n) -> n = 1/ri^2
    print('n_eff', N_eff)
    alpha = jnp.clip(N_eff / 20, 0.0, 1.0) # neff = 10 -> r = 1/sqrt(10)
    print('alpha = ', alpha)

n_eff 84.538185
alpha =  1.0
n_eff 10.97344
alpha =  0.548672
n_eff 20.973406
alpha =  1.0
n_eff 1.649811
alpha =  0.08249055
